In [1]:

# # Symbol Confusion Analysis for Composite DNA Decoding
# ## Analysis for A_11 alphabet, EZ17 error model, M=10

# =============================================================================
# CELL 1: DEVICE CONFIGURATION
# =============================================================================
import os
import torch

DEVICE_ID = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = DEVICE_ID


# =============================================================================
# CELL 2: IMPORTS
# =============================================================================
import random
import pickle
import json
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import time
from datetime import datetime
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")


# =============================================================================
# CELL 3: CONFIGURATION
# =============================================================================

# Fixed parameters for this analysis
ERROR_MODEL = "erlich"
ALPHABET_MODE = "2mix_3mix_4mix"  # A_11 alphabet
COVERAGE_M = 10

# Dataset parameters
NUM_SAMPLES = 100000
MAX_COVERAGE = 25

# Error model specifications
ERROR_MODEL_SPECS = {
    "erlich": {"seq_length": 136, "name": "EZ17"},
    "grass": {"seq_length": 104, "name": "G15"},
    "organick": {"seq_length": 77, "name": "O17"}
}

# Vocabulary sizes
VOCAB_SIZES = {
    "2mix_only": 10,
    "2mix_3mix": 14,
    "2mix_3mix_4mix": 15
}

# Build configuration
CONFIG = {
    "error_model": ERROR_MODEL,
    "error_name": ERROR_MODEL_SPECS[ERROR_MODEL]["name"],
    "alphabet_mode": ALPHABET_MODE,
    "coverage_M": COVERAGE_M,
    
    "dataset_dir": "./dataset",
    "dataset_name": f"dna_{ERROR_MODEL_SPECS[ERROR_MODEL]['name']}_{ALPHABET_MODE}",
    "results_dir": f"./results_{ERROR_MODEL_SPECS[ERROR_MODEL]['name']}_{ALPHABET_MODE}",
    
    "vocab_size": VOCAB_SIZES[ALPHABET_MODE],
    "seq_length": ERROR_MODEL_SPECS[ERROR_MODEL]["seq_length"],
    
    # Model Architecture (must match training)
    "input_channels": 4,
    "hidden_dim": 128,
    "num_layers": 2,
    "dropout": 0.2,
    "bidirectional": True,
    
    "batch_size": 500,
    "seed": 42
}

CONFIG["dataset_path"] = (f"{CONFIG['dataset_dir']}/"
                          f"{CONFIG['dataset_name']}_"
                          f"{NUM_SAMPLES}_{MAX_COVERAGE}.pkl")

# Output directory for confusion analysis
CONFIG["analysis_dir"] = f"./confusion_analysis_{CONFIG['error_name']}_{CONFIG['alphabet_mode']}"
os.makedirs(CONFIG['analysis_dir'], exist_ok=True)

print(f"{'='*60}")
print(f"📋 CONFUSION ANALYSIS CONFIGURATION")
print(f"{'='*60}")
print(f"   Error Model: {CONFIG['error_name']}")
print(f"   Alphabet: {CONFIG['alphabet_mode']} ({CONFIG['vocab_size']} classes)")
print(f"   Coverage M: {CONFIG['coverage_M']}")
print(f"   Sequence Length: {CONFIG['seq_length']}")
print(f"   Dataset Path: {CONFIG['dataset_path']}")
print(f"   Results Dir: {CONFIG['results_dir']}")
print(f"   Analysis Dir: {CONFIG['analysis_dir']}")
print(f"{'='*60}")


# =============================================================================
# CELL 4: SEED & REPRODUCIBILITY
# =============================================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(CONFIG['seed'])
print(f"🎲 Random seed set to: {CONFIG['seed']}")


# =============================================================================
# CELL 5: SYMBOL MAPPINGS & CLASSIFICATION
# =============================================================================

def build_symbol_to_idx(mode):
    """Build symbol-to-index mapping based on alphabet mode."""
    # Same as in training code - list only, copy from original
    symbol_to_idx = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    symbol_to_idx.update({'M1': 4, 'M2': 5, 'M3': 6, 'M4': 7, 'M5': 8, 'M6': 9})
    if mode in ["2mix_3mix", "2mix_3mix_4mix"]:
        symbol_to_idx.update({'T1': 10, 'T2': 11, 'T3': 12, 'T4': 13})
    if mode == "2mix_3mix_4mix":
        symbol_to_idx.update({'Q1': 14})
    return symbol_to_idx


def build_ideal_vectors(mode):
    """Build ideal frequency vectors for all symbols."""
    # Same as in training code - list only, copy from original
    ideal_vectors = [
        [1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0],
        [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0],
        [0.5, 0.0, 0.0, 0.5], [0.0, 0.5, 0.5, 0.0],
        [0.0, 0.5, 0.0, 0.5], [0.0, 0.0, 0.5, 0.5],
        [0.5, 0.5, 0.0, 0.0], [0.5, 0.0, 0.5, 0.0],
    ]
    if mode in ["2mix_3mix", "2mix_3mix_4mix"]:
        third = 1.0 / 3.0
        ideal_vectors.extend([
            [third, third, third, 0.0], [third, third, 0.0, third],
            [third, 0.0, third, third], [0.0, third, third, third],
        ])
    if mode == "2mix_3mix_4mix":
        ideal_vectors.append([0.25, 0.25, 0.25, 0.25])
    return torch.tensor(ideal_vectors, dtype=torch.float32)


def get_symbol_type(idx):
    """
    Classify symbol index into type category.
    
    Returns:
        str: 'Pure', '2-Mix', '3-Mix', or '4-Mix'
    """
    if idx <= 3:
        return 'Pure'
    elif idx <= 9:
        return '2-Mix'
    elif idx <= 13:
        return '3-Mix'
    else:
        return '4-Mix'


def get_symbol_composition(idx, symbol_to_idx, mode="2mix_3mix_4mix"):
    """
    Get the nucleotide composition for a symbol index.
    
    Returns:
        tuple: (symbol_name, composition_list)
    """
    idx_to_symbol = {v: k for k, v in symbol_to_idx.items()}
    symbol = idx_to_symbol[idx]
    
    compositions = {
        'A': ['A'], 'C': ['C'], 'G': ['G'], 'T': ['T'],
        'M1': ['A', 'T'], 'M2': ['C', 'G'], 'M3': ['C', 'T'],
        'M4': ['G', 'T'], 'M5': ['A', 'C'], 'M6': ['A', 'G'],
        'T1': ['A', 'C', 'G'], 'T2': ['A', 'C', 'T'],
        'T3': ['A', 'G', 'T'], 'T4': ['C', 'G', 'T'],
        'Q1': ['A', 'C', 'G', 'T']
    }
    
    return symbol, compositions.get(symbol, [])


# Build mappings
SYMBOL_TO_IDX = build_symbol_to_idx(CONFIG["alphabet_mode"])
IDX_TO_SYMBOL = {v: k for k, v in SYMBOL_TO_IDX.items()}
IDEAL_VECTORS = build_ideal_vectors(CONFIG["alphabet_mode"]).to(device)

# Define symbol type indices
PURE_INDICES = [0, 1, 2, 3]  # A, C, G, T
TWO_MIX_INDICES = [4, 5, 6, 7, 8, 9]  # M1-M6
THREE_MIX_INDICES = [10, 11, 12, 13]  # T1-T4
FOUR_MIX_INDICES = [14]  # Q1

print(f"\n📊 Symbol Classification:")
print(f"   Pure bases (0-3): {[IDX_TO_SYMBOL[i] for i in PURE_INDICES]}")
print(f"   2-Mix (4-9): {[IDX_TO_SYMBOL[i] for i in TWO_MIX_INDICES]}")
print(f"   3-Mix (10-13): {[IDX_TO_SYMBOL[i] for i in THREE_MIX_INDICES]}")
print(f"   4-Mix (14): {[IDX_TO_SYMBOL[i] for i in FOUR_MIX_INDICES]}")


# =============================================================================
# CELL 6: DATA PREPROCESSING
# =============================================================================

def preprocess_cluster_to_matrix(cluster_reads, target_length):
    """
    Convert variable-length noisy reads into a (4, target_length) normalized frequency matrix.
    Same as training code - list only, copy from original.
    """
    profile_matrix = np.zeros((4, target_length), dtype=np.float32)
    base_map = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    num_reads = len(cluster_reads)
    
    for read in cluster_reads:
        read_len = len(read)
        if read_len == 0:
            continue
        for t_idx in range(target_length):
            read_idx = int((t_idx + 0.5) * (read_len / target_length))
            if read_idx >= read_len:
                read_idx = read_len - 1
            base = read[read_idx]
            if base in base_map:
                row_idx = base_map[base]
                profile_matrix[row_idx, t_idx] += 1.0
    if num_reads > 0:
        profile_matrix /= num_reads
    return profile_matrix


# =============================================================================
# CELL 7: PYTORCH DATASET CLASS
# =============================================================================

class CompositeDNADataset(Dataset):
    """
    PyTorch Dataset for Composite DNA data.
    Same as training code - list only, copy from original.
    """
    def __init__(self, data_path, seq_length, symbol_to_idx, limit_coverage=None):
        with open(data_path, 'rb') as f:
            raw_data = pickle.load(f)
        self.samples = raw_data['data']
        self.metadata = raw_data['metadata']
        self.seq_length = seq_length
        self.symbol_to_idx = symbol_to_idx
        self.limit_coverage = limit_coverage
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        item = self.samples[idx]
        cluster = item['cluster']
        if self.limit_coverage is not None:
            actual_limit = min(self.limit_coverage, len(cluster))
            cluster = cluster[:actual_limit]
        x_data = preprocess_cluster_to_matrix(cluster, self.seq_length)
        label_seq = item['label']
        y_data = np.array([self.symbol_to_idx[s] for s in label_seq], dtype=np.longlong)
        return torch.tensor(x_data, dtype=torch.float32), torch.tensor(y_data, dtype=torch.long)


# =============================================================================
# CELL 8: NEURAL NETWORK MODEL
# =============================================================================

class CompositeDecoderLSTM(nn.Module):
    """
    Bidirectional LSTM Decoder for Composite DNA.
    Same as training code - list only, copy from original.
    """
    def __init__(self, config):
        super(CompositeDecoderLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=config['input_channels'],
            hidden_size=config['hidden_dim'],
            num_layers=config['num_layers'],
            batch_first=True,
            bidirectional=config['bidirectional'],
            dropout=config['dropout'] if config['num_layers'] > 1 else 0
        )
        fc_in = config['hidden_dim'] * 2 if config['bidirectional'] else config['hidden_dim']
        self.fc = nn.Linear(fc_in, config['vocab_size'])
        
    def forward(self, x):
        x = x.permute(0, 2, 1)
        out, _ = self.lstm(x)
        logits = self.fc(out)
        return logits.permute(0, 2, 1)


# =============================================================================
# CELL 9: BASELINE DECODERS
# =============================================================================

def min_distance_decoder(obs, ideal_vectors):
    """Minimum Euclidean Distance Decoder - list only, copy from original."""
    dists = torch.sum((obs.unsqueeze(2) - ideal_vectors.unsqueeze(0).unsqueeze(0)) ** 2, dim=3)
    return torch.argmin(dists, dim=2)


def kl_divergence_decoder(obs, ideal_vectors, epsilon=0.01):
    """KL Divergence Decoder - list only, copy from original."""
    ideal_safe = ideal_vectors.clone()
    ideal_safe = torch.clamp(ideal_safe, min=epsilon)
    ideal_safe = ideal_safe / ideal_safe.sum(dim=-1, keepdim=True)
    obs_expanded = obs.unsqueeze(2)
    log_ideal = torch.log(ideal_safe).unsqueeze(0).unsqueeze(0)
    cross_entropy = -(obs_expanded * log_ideal).sum(dim=-1)
    return torch.argmin(cross_entropy, dim=-1)


def maximum_likelihood_decoder(obs, ideal_vectors, epsilon=0.01):
    """Maximum Likelihood Decoder - list only, copy from original."""
    ideal_safe = ideal_vectors.clone()
    ideal_safe = torch.clamp(ideal_safe, min=epsilon)
    ideal_safe = ideal_safe / ideal_safe.sum(dim=-1, keepdim=True)
    obs_expanded = obs.unsqueeze(2)
    log_ideal = torch.log(ideal_safe).unsqueeze(0).unsqueeze(0)
    log_likelihood = (obs_expanded * log_ideal).sum(dim=-1)
    return torch.argmax(log_likelihood, dim=-1)


# =============================================================================
# CELL 10: CONFUSION MATRIX COMPUTATION
# =============================================================================

def compute_confusion_matrices(model, loader, ideal_vectors, device, num_classes):
    """
    Compute confusion matrices for all 4 decoders.
    
    Args:
        model: Trained Bi-LSTM model
        loader: Data loader
        ideal_vectors: Ideal frequency vectors for baseline decoders
        device: torch device
        num_classes: Number of symbol classes
    
    Returns:
        dict: Confusion matrices for each decoder {'lstm', 'mindist', 'kl', 'ml'}
              Each matrix is (num_classes x num_classes) where [i,j] = count of true=i, pred=j
    """
    model.eval()
    
    # Initialize confusion matrices
    confusion = {
        'lstm': np.zeros((num_classes, num_classes), dtype=np.int64),
        'mindist': np.zeros((num_classes, num_classes), dtype=np.int64),
        'kl': np.zeros((num_classes, num_classes), dtype=np.int64),
        'ml': np.zeros((num_classes, num_classes), dtype=np.int64)
    }
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            obs = inputs.permute(0, 2, 1)  # (Batch, L, 4)
            
            # Get predictions from all decoders
            outputs = model(inputs)
            pred_lstm = torch.argmax(outputs, dim=1)
            pred_mindist = min_distance_decoder(obs, ideal_vectors)
            pred_kl = kl_divergence_decoder(obs, ideal_vectors)
            pred_ml = maximum_likelihood_decoder(obs, ideal_vectors)
            
            # Flatten to 1D
            labels_flat = labels.view(-1).cpu().numpy()
            pred_lstm_flat = pred_lstm.view(-1).cpu().numpy()
            pred_mindist_flat = pred_mindist.view(-1).cpu().numpy()
            pred_kl_flat = pred_kl.view(-1).cpu().numpy()
            pred_ml_flat = pred_ml.view(-1).cpu().numpy()
            
            # Update confusion matrices
            for true_label, pred in zip(labels_flat, pred_lstm_flat):
                confusion['lstm'][true_label, pred] += 1
            for true_label, pred in zip(labels_flat, pred_mindist_flat):
                confusion['mindist'][true_label, pred] += 1
            for true_label, pred in zip(labels_flat, pred_kl_flat):
                confusion['kl'][true_label, pred] += 1
            for true_label, pred in zip(labels_flat, pred_ml_flat):
                confusion['ml'][true_label, pred] += 1
    
    return confusion


# =============================================================================
# CELL 11: CONFUSION ANALYSIS BY SYMBOL TYPE
# =============================================================================

def analyze_confusion_by_type(confusion_matrix, num_classes):
    """
    Analyze confusion patterns by symbol type.
    
    Args:
        confusion_matrix: (num_classes x num_classes) confusion matrix
        num_classes: Number of symbol classes
    
    Returns:
        dict: Confusion rates (%) for each type pair
    """
    # Define symbol type groups
    type_groups = {
        'Pure': PURE_INDICES,
        '2-Mix': TWO_MIX_INDICES,
        '3-Mix': THREE_MIX_INDICES,
        '4-Mix': FOUR_MIX_INDICES
    }
    
    # Initialize counters for each type pair
    type_pairs = [
        ('Pure', 'Pure'),
        ('Pure', '2-Mix'),
        ('2-Mix', '2-Mix'),
        ('2-Mix', '3-Mix'),
        ('3-Mix', '3-Mix'),
        ('3-Mix', '4-Mix'),
        ('Pure', '3-Mix'),
        ('Pure', '4-Mix'),
        ('2-Mix', '4-Mix'),
    ]
    
    confusion_counts = {pair: 0 for pair in type_pairs}
    total_counts = {pair: 0 for pair in type_pairs}
    
    # Calculate total samples per symbol
    total_per_symbol = confusion_matrix.sum(axis=1)
    
    for true_idx in range(num_classes):
        true_type = get_symbol_type(true_idx)
        true_count = total_per_symbol[true_idx]
        
        for pred_idx in range(num_classes):
            if true_idx == pred_idx:
                continue  # Skip correct predictions
                
            pred_type = get_symbol_type(pred_idx)
            count = confusion_matrix[true_idx, pred_idx]
            
            # Find matching type pair (order-independent)
            pair1 = (true_type, pred_type)
            pair2 = (pred_type, true_type)
            
            if pair1 in confusion_counts:
                confusion_counts[pair1] += count
            elif pair2 in confusion_counts:
                confusion_counts[pair2] += count
    
    # Calculate total samples for each type pair for normalization
    for true_type, true_indices in type_groups.items():
        for pred_type, pred_indices in type_groups.items():
            if true_type == pred_type:
                # Same type: count pairs within type (excluding diagonal)
                pair = (true_type, pred_type)
                if pair in total_counts:
                    for true_idx in true_indices:
                        total_counts[pair] += total_per_symbol[true_idx]
            else:
                # Different types
                pair1 = (true_type, pred_type)
                pair2 = (pred_type, true_type)
                if pair1 in total_counts:
                    for true_idx in true_indices:
                        total_counts[pair1] += total_per_symbol[true_idx]
                elif pair2 in total_counts:
                    for true_idx in true_indices:
                        total_counts[pair2] += total_per_symbol[true_idx]
    
    # Calculate percentages
    confusion_rates = {}
    for pair in type_pairs:
        if total_counts[pair] > 0:
            rate = 100.0 * confusion_counts[pair] / total_counts[pair]
        else:
            rate = 0.0
        confusion_rates[pair] = rate
    
    return confusion_rates, confusion_counts, total_counts


def analyze_confusion_by_type_v2(confusion_matrix, num_classes):
    """
    Alternative analysis: confusion rates as percentage of total errors for each type.
    
    For each type pair (A, B), calculates:
    - Number of times a symbol of type A was confused with a symbol of type B (bidirectional)
    - As a percentage of total predictions for symbols of type A
    
    Returns:
        dict: Confusion rates (%) for key type pairs
    """
    total_predictions = confusion_matrix.sum()
    total_errors = total_predictions - np.trace(confusion_matrix)
    
    results = {}
    
    # Key confusion pairs to analyze
    pairs_to_analyze = [
        ('Pure', 'Pure'),
        ('Pure', '2-Mix'),
        ('2-Mix', '2-Mix'),
        ('2-Mix', '3-Mix'),
        ('3-Mix', '3-Mix'),
    ]
    
    type_to_indices = {
        'Pure': PURE_INDICES,
        '2-Mix': TWO_MIX_INDICES,
        '3-Mix': THREE_MIX_INDICES,
        '4-Mix': FOUR_MIX_INDICES
    }
    
    for type_a, type_b in pairs_to_analyze:
        indices_a = type_to_indices[type_a]
        indices_b = type_to_indices[type_b]
        
        # Count confusions: true=type_a, pred=type_b (excluding correct when same type)
        confusion_count = 0
        total_for_type_a = 0
        
        for true_idx in indices_a:
            total_for_type_a += confusion_matrix[true_idx, :].sum()
            for pred_idx in indices_b:
                if true_idx != pred_idx:  # Exclude correct predictions
                    confusion_count += confusion_matrix[true_idx, pred_idx]
        
        # Also count reverse direction for bidirectional pairs
        if type_a != type_b:
            for true_idx in indices_b:
                for pred_idx in indices_a:
                    confusion_count += confusion_matrix[true_idx, pred_idx]
        
        # Calculate as percentage of predictions for type A symbols
        if total_for_type_a > 0:
            rate = 100.0 * confusion_count / total_for_type_a
        else:
            rate = 0.0
        
        pair_key = f"{type_a} ↔ {type_b}"
        results[pair_key] = {
            'rate': rate,
            'count': confusion_count,
            'total': total_for_type_a
        }
    
    return results


# =============================================================================
# CELL 12: TOP CONFUSED PAIRS ANALYSIS
# =============================================================================

def get_top_confused_pairs(confusion_matrix, idx_to_symbol, top_k=10):
    """
    Identify the most frequently confused symbol pairs.
    
    Args:
        confusion_matrix: (num_classes x num_classes) confusion matrix
        idx_to_symbol: Mapping from index to symbol name
        top_k: Number of top pairs to return
    
    Returns:
        list: List of tuples (true_symbol, pred_symbol, count, rate)
    """
    num_classes = confusion_matrix.shape[0]
    pairs = []
    
    for true_idx in range(num_classes):
        total_for_true = confusion_matrix[true_idx, :].sum()
        for pred_idx in range(num_classes):
            if true_idx != pred_idx:
                count = confusion_matrix[true_idx, pred_idx]
                if count > 0:
                    rate = 100.0 * count / total_for_true if total_for_true > 0 else 0
                    pairs.append({
                        'true_idx': true_idx,
                        'pred_idx': pred_idx,
                        'true_symbol': idx_to_symbol[true_idx],
                        'pred_symbol': idx_to_symbol[pred_idx],
                        'true_type': get_symbol_type(true_idx),
                        'pred_type': get_symbol_type(pred_idx),
                        'count': count,
                        'rate': rate
                    })
    
    # Sort by count descending
    pairs.sort(key=lambda x: x['count'], reverse=True)
    return pairs[:top_k]


def compute_symbol_pair_distance(idx1, idx2, ideal_vectors):
    """Compute Euclidean distance between two symbols' ideal vectors."""
    vec1 = ideal_vectors[idx1].cpu().numpy()
    vec2 = ideal_vectors[idx2].cpu().numpy()
    return np.linalg.norm(vec1 - vec2)


# =============================================================================
# CELL 13: ADJACENT SYMBOL CONFUSION ANALYSIS
# =============================================================================

def analyze_adjacent_confusions(confusion_matrices, ideal_vectors, idx_to_symbol, distance_threshold=0.5):
    """
    Analyze confusions between symbols with similar ideal frequency vectors.
    
    Args:
        confusion_matrices: Dict of confusion matrices for each decoder
        ideal_vectors: Ideal frequency vectors
        idx_to_symbol: Mapping from index to symbol name
        distance_threshold: Max distance to consider symbols "adjacent"
    
    Returns:
        dict: Analysis results for each decoder
    """
    num_classes = len(idx_to_symbol)
    
    # Find adjacent symbol pairs (small distance)
    adjacent_pairs = []
    for i in range(num_classes):
        for j in range(i+1, num_classes):
            dist = compute_symbol_pair_distance(i, j, ideal_vectors)
            if dist < distance_threshold:
                adjacent_pairs.append({
                    'idx1': i, 'idx2': j,
                    'sym1': idx_to_symbol[i], 'sym2': idx_to_symbol[j],
                    'distance': dist
                })
    
    print(f"\n📏 Found {len(adjacent_pairs)} adjacent symbol pairs (distance < {distance_threshold}):")
    for pair in sorted(adjacent_pairs, key=lambda x: x['distance'])[:10]:
        print(f"   {pair['sym1']} ↔ {pair['sym2']}: distance = {pair['distance']:.4f}")
    
    # Count confusions for adjacent pairs per decoder
    results = {}
    for decoder_name, conf_matrix in confusion_matrices.items():
        adjacent_confusion_count = 0
        total_adjacent_samples = 0
        
        for pair in adjacent_pairs:
            i, j = pair['idx1'], pair['idx2']
            # Bidirectional confusion
            adjacent_confusion_count += conf_matrix[i, j] + conf_matrix[j, i]
            total_adjacent_samples += conf_matrix[i, :].sum() + conf_matrix[j, :].sum()
        
        results[decoder_name] = {
            'adjacent_confusion_count': adjacent_confusion_count,
            'total_adjacent_samples': total_adjacent_samples,
            'adjacent_confusion_rate': 100.0 * adjacent_confusion_count / total_adjacent_samples if total_adjacent_samples > 0 else 0
        }
    
    return results, adjacent_pairs


# =============================================================================
# CELL 14: VISUALIZATION FUNCTIONS
# =============================================================================

def plot_confusion_matrix(confusion_matrix, idx_to_symbol, decoder_name, save_path):
    """Plot and save confusion matrix heatmap."""
    num_classes = confusion_matrix.shape[0]
    labels = [idx_to_symbol[i] for i in range(num_classes)]
    
    # Normalize by row (true labels)
    row_sums = confusion_matrix.sum(axis=1, keepdims=True)
    normalized = confusion_matrix / (row_sums + 1e-10) * 100
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(normalized, annot=True, fmt='.1f', cmap='Blues',
                xticklabels=labels, yticklabels=labels,
                cbar_kws={'label': 'Percentage (%)'})
    plt.xlabel('Predicted Symbol', fontsize=12)
    plt.ylabel('True Symbol', fontsize=12)
    plt.title(f'Confusion Matrix: {decoder_name}\n(Row-normalized percentages)', fontsize=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   💾 Saved: {save_path}")


def plot_type_confusion_comparison(type_results, save_path):
    """Plot bar chart comparing confusion rates by type across decoders."""
    decoders = list(type_results.keys())
    
    # Key pairs to plot
    pairs = ['Pure ↔ Pure', 'Pure ↔ 2-Mix', '2-Mix ↔ 2-Mix', 
             '2-Mix ↔ 3-Mix', '3-Mix ↔ 3-Mix']
    
    x = np.arange(len(pairs))
    width = 0.2
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    colors = {'lstm': '#2ecc71', 'kl': '#3498db', 'ml': '#9b59b6', 'mindist': '#e74c3c'}
    
    for i, decoder in enumerate(decoders):
        rates = [type_results[decoder].get(pair, {}).get('rate', 0) for pair in pairs]
        offset = (i - len(decoders)/2 + 0.5) * width
        bars = ax.bar(x + offset, rates, width, label=decoder.upper(), color=colors.get(decoder, 'gray'))
        
        # Add value labels on bars
        for bar, rate in zip(bars, rates):
            if rate > 0.1:
                ax.annotate(f'{rate:.1f}',
                           xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                           xytext=(0, 3), textcoords="offset points",
                           ha='center', va='bottom', fontsize=8)
    
    ax.set_xlabel('Confusion Type', fontsize=12)
    ax.set_ylabel('Confusion Rate (%)', fontsize=12)
    ax.set_title('Symbol Type Confusion Rates by Decoder', fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels(pairs, rotation=15, ha='right')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   💾 Saved: {save_path}")


# =============================================================================
# CELL 15: LOAD DATA AND MODEL
# =============================================================================

print("\n" + "="*70)
print("📦 LOADING DATA AND MODEL")
print("="*70)

# Check dataset exists
if not os.path.exists(CONFIG['dataset_path']):
    raise FileNotFoundError(f"Dataset not found: {CONFIG['dataset_path']}")

# Load dataset
full_ds = CompositeDNADataset(
    CONFIG['dataset_path'], 
    CONFIG['seq_length'],
    SYMBOL_TO_IDX,
    limit_coverage=CONFIG['coverage_M']
)

# Split into train/test (same split as training)
set_seed(CONFIG['seed'])
train_size = int(0.8 * len(full_ds))
val_size = len(full_ds) - train_size
train_ds, val_ds = random_split(full_ds, [train_size, val_size])

# Create test loader
test_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0)

print(f"✅ Dataset loaded: {len(full_ds):,} total samples")
print(f"   Test set: {val_size:,} samples")
print(f"   Coverage: M = {CONFIG['coverage_M']}")

# Load trained model
model = CompositeDecoderLSTM(CONFIG).to(device)

model_prefix = f"{CONFIG['error_name']}_{CONFIG['alphabet_mode']}"
best_weights_path = os.path.join(CONFIG['results_dir'], f"best_model_{model_prefix}_M{CONFIG['coverage_M']}.pth")

if not os.path.exists(best_weights_path):
    raise FileNotFoundError(f"Model weights not found: {best_weights_path}")

model.load_state_dict(torch.load(best_weights_path, map_location=device))
model.eval()

num_params = sum(p.numel() for p in model.parameters())
print(f"✅ Model loaded: {best_weights_path}")
print(f"   Parameters: {num_params:,}")


# =============================================================================
# CELL 16: COMPUTE CONFUSION MATRICES
# =============================================================================

print("\n" + "="*70)
print("🔄 COMPUTING CONFUSION MATRICES")
print("="*70)

start_time = time.time()
confusion_matrices = compute_confusion_matrices(
    model, test_loader, IDEAL_VECTORS, device, CONFIG['vocab_size']
)
elapsed = time.time() - start_time

print(f"✅ Confusion matrices computed in {elapsed:.1f}s")

# Verify accuracy matches expected
for decoder_name, conf_matrix in confusion_matrices.items():
    correct = np.trace(conf_matrix)
    total = conf_matrix.sum()
    accuracy = 100.0 * correct / total
    print(f"   {decoder_name.upper()}: Accuracy = {accuracy:.2f}%")


# =============================================================================
# CELL 17: ANALYZE CONFUSION BY SYMBOL TYPE
# =============================================================================

print("\n" + "="*70)
print("📊 CONFUSION ANALYSIS BY SYMBOL TYPE")
print("="*70)

type_results = {}
for decoder_name, conf_matrix in confusion_matrices.items():
    type_results[decoder_name] = analyze_confusion_by_type_v2(conf_matrix, CONFIG['vocab_size'])

# Print results in table format
pairs_to_show = ['Pure ↔ Pure', 'Pure ↔ 2-Mix', '2-Mix ↔ 2-Mix', '2-Mix ↔ 3-Mix', '3-Mix ↔ 3-Mix']

print(f"\n{'Confusion Type':<20} {'Bi-LSTM':>10} {'KL/ML':>10} {'Min.D':>10}")
print("-" * 52)

for pair in pairs_to_show:
    lstm_rate = type_results['lstm'].get(pair, {}).get('rate', 0)
    kl_rate = type_results['kl'].get(pair, {}).get('rate', 0)
    mindist_rate = type_results['mindist'].get(pair, {}).get('rate', 0)
    print(f"{pair:<20} {lstm_rate:>10.2f} {kl_rate:>10.2f} {mindist_rate:>10.2f}")

# Save to JSON
def convert_to_serializable(obj):
    """Convert numpy types to Python native types for JSON serialization."""
    if isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(item) for item in obj]
    elif isinstance(obj, (np.integer, np.int64, np.int32)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64, np.float32)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    else:
        return obj

type_results_path = os.path.join(CONFIG['analysis_dir'], 'confusion_by_type.json')
with open(type_results_path, 'w') as f:
    json.dump(convert_to_serializable(type_results), f, indent=4)
print(f"\n💾 Results saved: {type_results_path}")


# =============================================================================
# CELL 18: TOP CONFUSED PAIRS
# =============================================================================

print("\n" + "="*70)
print("🔝 TOP CONFUSED SYMBOL PAIRS")
print("="*70)

top_pairs_results = {}
for decoder_name, conf_matrix in confusion_matrices.items():
    top_pairs = get_top_confused_pairs(conf_matrix, IDX_TO_SYMBOL, top_k=15)
    top_pairs_results[decoder_name] = top_pairs
    
    print(f"\n📌 {decoder_name.upper()} - Top 10 Confused Pairs:")
    print(f"   {'True':<6} {'Pred':<6} {'Type→Type':<15} {'Count':>8} {'Rate':>8}")
    print("   " + "-" * 50)
    for pair in top_pairs[:10]:
        type_str = f"{pair['true_type']}→{pair['pred_type']}"
        print(f"   {pair['true_symbol']:<6} {pair['pred_symbol']:<6} {type_str:<15} {pair['count']:>8} {pair['rate']:>7.2f}%")

# Save to JSON
top_pairs_path = os.path.join(CONFIG['analysis_dir'], 'top_confused_pairs.json')
with open(top_pairs_path, 'w') as f:
    json.dump(convert_to_serializable(top_pairs_results), f, indent=4)
print(f"\n💾 Results saved: {top_pairs_path}")


# =============================================================================
# CELL 19: ADJACENT SYMBOL CONFUSION ANALYSIS
# =============================================================================

print("\n" + "="*70)
print("📏 ADJACENT SYMBOL CONFUSION ANALYSIS")
print("="*70)

adjacent_results, adjacent_pairs = analyze_adjacent_confusions(
    confusion_matrices, IDEAL_VECTORS, IDX_TO_SYMBOL, distance_threshold=0.5
)

print(f"\n{'Decoder':<12} {'Adjacent Confusions':>20} {'Rate':>10}")
print("-" * 45)
for decoder_name, results in adjacent_results.items():
    print(f"{decoder_name.upper():<12} {results['adjacent_confusion_count']:>20} {results['adjacent_confusion_rate']:>9.2f}%")

# Calculate reduction percentages
kl_adj_rate = adjacent_results['kl']['adjacent_confusion_rate']
lstm_adj_rate = adjacent_results['lstm']['adjacent_confusion_rate']
adj_reduction = 0.0
if kl_adj_rate > 0:
    adj_reduction = 100.0 * (kl_adj_rate - lstm_adj_rate) / kl_adj_rate
    print(f"\n🎯 Bi-LSTM reduces adjacent-symbol confusions by {adj_reduction:.1f}% relative to KL/ML")

# Save results
adjacent_results_path = os.path.join(CONFIG['analysis_dir'], 'adjacent_confusion_analysis.json')
save_data = {
    'results': convert_to_serializable(adjacent_results),
    'adjacent_pairs': convert_to_serializable(adjacent_pairs),
    'reduction_vs_kl_percent': float(adj_reduction)
}
with open(adjacent_results_path, 'w') as f:
    json.dump(save_data, f, indent=4)
print(f"💾 Results saved: {adjacent_results_path}")


# =============================================================================
# CELL 20: SPECIFIC PAIR ANALYSIS
# =============================================================================

print("\n" + "="*70)
print("🔍 SPECIFIC PAIR ANALYSIS: Similar Composition Pairs")
print("="*70)

# Key pairs to analyze (symbols with partial overlap)
analysis_pairs = [
    (8, 11, "M5 (A|C)", "T2 (A|C|T)"),   # M5 vs T2
    (9, 10, "M6 (A|G)", "T1 (A|C|G)"),   # M6 vs T1
    (4, 12, "M1 (A|T)", "T3 (A|G|T)"),   # M1 vs T3
    (5, 13, "M2 (C|G)", "T4 (C|G|T)"),   # M2 vs T4
    (10, 14, "T1 (A|C|G)", "Q1 (A|C|G|T)"),  # T1 vs Q1
]

print(f"\n   Analyzing pairs with overlapping nucleotide compositions:\n")

for idx1, idx2, name1, name2 in analysis_pairs:
    distance = compute_symbol_pair_distance(idx1, idx2, IDEAL_VECTORS)
    print(f"   {name1} vs {name2}")
    print(f"   Distance: {distance:.4f}")
    print(f"   {'Decoder':<12} {name1[:5]+'→'+name2[:5]:>15} {name2[:5]+'→'+name1[:5]:>15} {'Total':>10}")
    print("   " + "-" * 55)
    
    for decoder_name, conf_matrix in confusion_matrices.items():
        c1_to_c2 = conf_matrix[idx1, idx2]
        c2_to_c1 = conf_matrix[idx2, idx1]
        total = c1_to_c2 + c2_to_c1
        print(f"   {decoder_name.upper():<12} {c1_to_c2:>15} {c2_to_c1:>15} {total:>10}")
    print()


# =============================================================================
# CELL 21: GENERATE VISUALIZATIONS
# =============================================================================

print("\n" + "="*70)
print("📈 GENERATING VISUALIZATIONS")
print("="*70)

# Plot confusion matrices for each decoder
for decoder_name, conf_matrix in confusion_matrices.items():
    save_path = os.path.join(CONFIG['analysis_dir'], f'confusion_matrix_{decoder_name}.png')
    plot_confusion_matrix(conf_matrix, IDX_TO_SYMBOL, decoder_name.upper(), save_path)

# Plot type confusion comparison
comparison_path = os.path.join(CONFIG['analysis_dir'], 'type_confusion_comparison.png')
plot_type_confusion_comparison(type_results, comparison_path)


# =============================================================================
# CELL 22: GENERATE LATEX TABLE FOR PAPER
# =============================================================================

print("\n" + "="*70)
print("📝 LATEX TABLE OUTPUT")
print("="*70)

latex_table = r"""
\begin{table}[h]
\caption{Confusion rates (\%) by symbol type for $\mathcal{A}_{11}$/EZ17 at $M=10$}
\centering
\begin{tabular}{l|ccc}
\toprule
\textbf{Confusion Type} & \textbf{Bi-LSTM} & \textbf{KL/ML} & \textbf{Min.D} \\
\midrule
"""

for pair in pairs_to_show:
    lstm_rate = type_results['lstm'].get(pair, {}).get('rate', 0)
    kl_rate = type_results['kl'].get(pair, {}).get('rate', 0)
    mindist_rate = type_results['mindist'].get(pair, {}).get('rate', 0)
    
    latex_pair = pair.replace('↔', r'$\leftrightarrow$')
    latex_table += f"{latex_pair} & {lstm_rate:.1f} & {kl_rate:.1f} & {mindist_rate:.1f} \\\\\n"

latex_table += r"""\bottomrule
\end{tabular}
\label{Table:ConfusionByType}
\end{table}
"""

print(latex_table)

latex_path = os.path.join(CONFIG['analysis_dir'], 'confusion_table.tex')
with open(latex_path, 'w') as f:
    f.write(latex_table)
print(f"💾 LaTeX table saved: {latex_path}")


# =============================================================================
# CELL 23: CALCULATE KEY METRICS FOR PAPER TEXT
# =============================================================================

print("\n" + "="*70)
print("📊 KEY METRICS FOR PAPER TEXT")
print("="*70)

# Calculate reduction percentages for different confusion types
print("\n1. REDUCTION PERCENTAGES (Bi-LSTM vs KL/ML):")
print("-" * 50)

reductions = {}
for pair in pairs_to_show:
    lstm_rate = type_results['lstm'].get(pair, {}).get('rate', 0)
    kl_rate = type_results['kl'].get(pair, {}).get('rate', 0)
    if kl_rate > 0:
        reduction = 100.0 * (kl_rate - lstm_rate) / kl_rate
    else:
        reduction = 0.0 if lstm_rate == 0 else -100.0
    reductions[pair] = reduction
    print(f"   {pair}: {reduction:.1f}% reduction")

print("\n2. ADJACENT SYMBOL CONFUSION:")
print("-" * 50)
print(f"   Bi-LSTM rate: {adjacent_results['lstm']['adjacent_confusion_rate']:.2f}%")
print(f"   KL/ML rate: {adjacent_results['kl']['adjacent_confusion_rate']:.2f}%")
print(f"   Reduction: {adj_reduction:.1f}%")

pure_2mix_reduction = reductions.get('Pure ↔ 2-Mix', 0)
mix2_3mix_reduction = reductions.get('2-Mix ↔ 3-Mix', 0)
print(f"\n3. KEY FINDINGS FOR PAPER:")
print("-" * 50)
print(f"   • Pure ↔ 2-Mix confusion reduction: {pure_2mix_reduction:.1f}%")
print(f"   • 2-Mix ↔ 3-Mix confusion reduction: {mix2_3mix_reduction:.1f}%")

# Filter out zero/negative reductions for range calculation
positive_reductions = [r for r in reductions.values() if r > 0]
if positive_reductions:
    print(f"   • Range of reductions: {min(positive_reductions):.0f}% - {max(positive_reductions):.0f}%")


# =============================================================================
# CELL 24: SUMMARY STATISTICS
# =============================================================================

print("\n" + "="*70)
print("📊 SUMMARY STATISTICS")
print("="*70)

summary = {
    'config': {
        'error_model': CONFIG['error_name'],
        'alphabet': CONFIG['alphabet_mode'],
        'vocab_size': CONFIG['vocab_size'],
        'coverage_M': CONFIG['coverage_M'],
        'test_samples': val_size
    },
    'accuracy': {},
    'type_confusion_rates': {},
    'adjacent_confusion': {},
    'reduction_percentages': {}
}

for decoder_name, conf_matrix in confusion_matrices.items():
    accuracy = 100.0 * np.trace(conf_matrix) / conf_matrix.sum()
    summary['accuracy'][decoder_name] = float(accuracy)

for decoder_name in confusion_matrices.keys():
    summary['type_confusion_rates'][decoder_name] = {
        pair: float(type_results[decoder_name].get(pair, {}).get('rate', 0))
        for pair in pairs_to_show
    }

for decoder_name, results in adjacent_results.items():
    summary['adjacent_confusion'][decoder_name] = float(results['adjacent_confusion_rate'])

summary['reduction_percentages'] = {k: float(v) for k, v in reductions.items()}

print(f"\n🎯 Key Findings:")
print(f"\n   1. Accuracy at M={CONFIG['coverage_M']}:")
for decoder, acc in summary['accuracy'].items():
    print(f"      {decoder.upper()}: {acc:.2f}%")

print(f"\n   2. Bi-LSTM Reduction vs KL/ML by Confusion Type:")
for pair, red in summary['reduction_percentages'].items():
    print(f"      {pair}: {red:.1f}% reduction")

print(f"\n   3. Adjacent Symbol Confusion Rates:")
for decoder, rate in summary['adjacent_confusion'].items():
    print(f"      {decoder.upper()}: {rate:.2f}%")

summary_path = os.path.join(CONFIG['analysis_dir'], 'analysis_summary.json')
with open(summary_path, 'w') as f:
    json.dump(convert_to_serializable(summary), f, indent=4)
print(f"\n💾 Summary saved: {summary_path}")

print("\n" + "="*70)
print("✅ CONFUSION ANALYSIS COMPLETE")
print(f"   All results saved to: {CONFIG['analysis_dir']}")
print("="*70)

✅ Using device: cuda
   GPU: NVIDIA GeForce RTX 3080
📋 CONFUSION ANALYSIS CONFIGURATION
   Error Model: EZ17
   Alphabet: 2mix_3mix_4mix (15 classes)
   Coverage M: 10
   Sequence Length: 136
   Dataset Path: ./dataset/dna_EZ17_2mix_3mix_4mix_100000_25.pkl
   Results Dir: ./results_EZ17_2mix_3mix_4mix
   Analysis Dir: ./confusion_analysis_EZ17_2mix_3mix_4mix
🎲 Random seed set to: 42

📊 Symbol Classification:
   Pure bases (0-3): ['A', 'C', 'G', 'T']
   2-Mix (4-9): ['M1', 'M2', 'M3', 'M4', 'M5', 'M6']
   3-Mix (10-13): ['T1', 'T2', 'T3', 'T4']
   4-Mix (14): ['Q1']

📦 LOADING DATA AND MODEL
✅ Dataset loaded: 100,000 total samples
   Test set: 20,000 samples
   Coverage: M = 10
✅ Model loaded: ./results_EZ17_2mix_3mix_4mix/best_model_EZ17_2mix_3mix_4mix_M10.pth
   Parameters: 536,335

🔄 COMPUTING CONFUSION MATRICES
✅ Confusion matrices computed in 57.4s
   LSTM: Accuracy = 92.48%
   MINDIST: Accuracy = 80.68%
   KL: Accuracy = 84.69%
   ML: Accuracy = 84.69%

📊 CONFUSION ANALYSIS BY SYM

   💾 Saved: ./confusion_analysis_EZ17_2mix_3mix_4mix/confusion_matrix_lstm.png
   💾 Saved: ./confusion_analysis_EZ17_2mix_3mix_4mix/confusion_matrix_mindist.png
   💾 Saved: ./confusion_analysis_EZ17_2mix_3mix_4mix/confusion_matrix_kl.png
   💾 Saved: ./confusion_analysis_EZ17_2mix_3mix_4mix/confusion_matrix_ml.png
   💾 Saved: ./confusion_analysis_EZ17_2mix_3mix_4mix/type_confusion_comparison.png

📝 LATEX TABLE OUTPUT

\begin{table}[h]
\caption{Confusion rates (\%) by symbol type for $\mathcal{A}_{11}$/EZ17 at $M=10$}
\centering
\begin{tabular}{l|ccc}
\toprule
\textbf{Confusion Type} & \textbf{Bi-LSTM} & \textbf{KL/ML} & \textbf{Min.D} \\
\midrule
Pure $\leftrightarrow$ Pure & 0.0 & 0.0 & 0.0 \\
Pure $\leftrightarrow$ 2-Mix & 3.3 & 7.4 & 16.2 \\
2-Mix $\leftrightarrow$ 2-Mix & 0.2 & 0.2 & 0.2 \\
2-Mix $\leftrightarrow$ 3-Mix & 9.3 & 20.5 & 21.9 \\
3-Mix $\leftrightarrow$ 3-Mix & 0.7 & 0.1 & 0.7 \\
\bottomrule
\end{tabular}
\label{Table:ConfusionByType}
\end{table}

💾 LaTeX table saved: .